# Step 3 — Feature Engineering
**Course:** ETL (G01) — Workshop 3  
**Goal:** Select, analyze and preprocess features for the regression model.

### Feature selection decisions (justified from real data)

| Feature | Correlation with `happiness_score` | Decision | Justification |
|---------|-----------------------------------|----------|---------------|
| `gdp` | **0.789** | ✅ Keep | Strongest predictor |
| `health` | **0.743** | ✅ Keep | Strong predictor |
| `family` | **0.649** | ✅ Keep | Strong predictor |
| `freedom` | **0.551** | ✅ Keep | Moderate-strong predictor |
| `corruption` | **0.397** | ✅ Keep | Moderate predictor, meaningful signal |
| `generosity` | **0.138** | ✅ Keep | Weak but included — all features are part of the Kafka event schema |
| `year` | — | ❌ Drop | Not a happiness predictor, just a data identifier |
| `country` | — | ❌ Drop | Categorical with 150+ values, not needed for this simple regression |

**Scaling decision:** Features have different std deviations (0.10 – 0.41).  
We apply `StandardScaler` to normalize them for the regression model.

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
import joblib
import os

sns.set_theme(style='whitegrid', palette='muted')
pd.set_option('display.max_columns', None)

## 2. Load Unified Dataset

In [ ]:
# ------------------------------------------------------------------
# Load the unified dataset produced by cleaning.ipynb
# ------------------------------------------------------------------
df = pd.read_csv('../data/processed/happiness_unified.csv')

print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
display(df.head())

## 3. Descriptive Statistics per Feature

In [ ]:
# ------------------------------------------------------------------
# Overview of numeric features
# ------------------------------------------------------------------
features = ['gdp', 'family', 'health', 'freedom', 'generosity', 'corruption']
target   = 'happiness_score'

display(df[features + [target]].describe().round(4))

## 4. Correlation Analysis

In [ ]:
# ------------------------------------------------------------------
# Correlation of each feature with the target (happiness_score)
# ------------------------------------------------------------------
corr_with_target = (
    df[features + [target]]
    .corr()[target]
    .drop(target)
    .sort_values(ascending=False)
)

print('Correlation with happiness_score:\n')
print(corr_with_target.round(4))

# Bar chart
plt.figure(figsize=(8, 4))
corr_with_target.plot(kind='bar', color='steelblue', edgecolor='white')
plt.title('Feature Correlation with Happiness Score')
plt.ylabel('Pearson Correlation')
plt.xticks(rotation=30)
plt.axhline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()

## 5. Feature Distributions

In [ ]:
# ------------------------------------------------------------------
# Histogram for each feature to check distribution shape
# ------------------------------------------------------------------
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

for i, feat in enumerate(features):
    axes[i].hist(df[feat], bins=30, color='steelblue', edgecolor='white')
    axes[i].set_title(feat)
    axes[i].set_xlabel('Value')
    axes[i].set_ylabel('Count')

plt.suptitle('Feature Distributions — Unified Dataset', fontsize=14)
plt.tight_layout()
plt.show()

## 6. Scatter Plots — Features vs Target

In [ ]:
# ------------------------------------------------------------------
# Scatter plot of each feature against happiness_score
# Helps visualize linear relationships before modeling
# ------------------------------------------------------------------
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

for i, feat in enumerate(features):
    axes[i].scatter(df[feat], df[target], alpha=0.3, color='steelblue', s=10)
    axes[i].set_xlabel(feat)
    axes[i].set_ylabel('happiness_score')
    axes[i].set_title(f'{feat} vs happiness_score')

plt.suptitle('Feature vs Target Scatter Plots', fontsize=14)
plt.tight_layout()
plt.show()

## 7. Pairplot — All Features

In [ ]:
# ------------------------------------------------------------------
# Pairplot to explore relationships between all features
# ------------------------------------------------------------------
sns.pairplot(
    df[features + [target]],
    plot_kws={'alpha': 0.2, 's': 10},
    diag_kind='kde'
)
plt.suptitle('Pairplot — All Features + Target', y=1.01, fontsize=13)
plt.tight_layout()
plt.show()

## 8. Feature Scaling with StandardScaler

In [ ]:
# ------------------------------------------------------------------
# Scale features using StandardScaler
# Reason: features have different std devs (0.10 to 0.41),
# scaling ensures no feature dominates due to magnitude differences.
# The scaler is fitted on ALL data here and saved for the consumer.
# The train/test split happens in model_training.ipynb.
# ------------------------------------------------------------------
X = df[features].copy()
y = df[target].copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=features)

print('Before scaling:')
display(X.describe().round(4))

print('\nAfter scaling (mean ≈ 0, std ≈ 1):')
display(X_scaled.describe().round(4))

## 9. Save Feature-Engineered Dataset and Scaler

In [ ]:
# ------------------------------------------------------------------
# Build the final ML-ready dataset:
# - Scaled features
# - Target column (unscaled — we predict raw happiness score)
# - country and year kept for traceability (not used in training)
# ------------------------------------------------------------------
df_ml = X_scaled.copy()
df_ml['happiness_score'] = y.values
df_ml['country']         = df['country'].values
df_ml['year']            = df['year'].values

# Reorder: traceability columns first, then features, then target
df_ml = df_ml[['country', 'year'] + features + ['happiness_score']]

# Save ML-ready dataset
ml_path = '../data/processed/happiness_ml_ready.csv'
df_ml.to_csv(ml_path, index=False)
print(f'ML-ready dataset saved to: {ml_path}')
print(f'Shape: {df_ml.shape}')

# Save scaler — the Kafka consumer will use this to scale incoming events
os.makedirs('../models', exist_ok=True)
scaler_path = '../models/scaler.pkl'
joblib.dump(scaler, scaler_path)
print(f'Scaler saved to: {scaler_path}')

## 10. Feature Engineering Summary

In [ ]:
# ------------------------------------------------------------------
# Print a final summary of all decisions made
# ------------------------------------------------------------------
summary = {
    'Selected features': features,
    'Target': target,
    'Dropped columns': ['country (not used in training)', 'year (not used in training)'],
    'Scaling applied': 'StandardScaler (mean=0, std=1)',
    'Scaling justification': 'Features have different magnitudes (std from 0.10 to 0.41)',
    'Target leakage avoided': 'happiness_score is only the target, never a feature',
    'Categorical handling': 'country dropped — 150+ unique values, not needed for simple regression',
    'Outputs': [
        '../data/processed/happiness_ml_ready.csv',
        '../models/scaler.pkl',
    ],
}

for key, value in summary.items():
    if isinstance(value, list):
        print(f'\n{key}:')
        for v in value:
            print(f'  - {v}')
    else:
        print(f'\n{key}: {value}')